# Datos recibidos de Moodle vía LTI

Este cuadernillo muestra los datos del lanzamiento LTI tal como llegan a JupyterHub. No los lee directamente del protocolo LTI (eso lo hace el `LTI11Authenticator` en `jupyterhub_config.py`) — los lee de las variables de entorno que el `auth_state_hook` deja listas en tu servidor al arrancar tu sesión.

**Para que aparezcan datos reales, este cuadernillo tiene que abrirse desde el enlace de la actividad en Moodle**, no entrando directo a JupyterHub. Si lo corres fuera de ese flujo (por ejemplo probando en local), vas a ver "(no disponible)" en todos los campos — es el comportamiento esperado, no un error.

In [ ]:
import os

# Nombres de variables que configuramos en el auth_state_hook de jupyterhub_config.py
VARIABLES_LTI = {
    "ID de usuario (user_id)": "ALUMNO_ID",
    "Nombre completo": "ALUMNO_NOMBRE",
    "Correo": "ALUMNO_EMAIL",
    "Rol en el curso": "ALUMNO_ROL",
    "ID del curso": "CURSO_ID",
    "Nombre del curso": "CURSO_NOMBRE",
}

datos_lti = {etiqueta: os.environ.get(var, "") for etiqueta, var in VARIABLES_LTI.items()}
hay_datos = any(valor for valor in datos_lti.values())

if not hay_datos:
    print("No se encontraron datos de LTI en las variables de entorno.")
    print("¿Abriste este cuadernillo desde el enlace de la actividad en Moodle?")
else:
    for etiqueta, valor in datos_lti.items():
        print(f"{etiqueta}: {valor if valor else '(no disponible)'}")

## Vista más legible

In [ ]:
from IPython.display import Markdown, display

filas = "\n".join(f"| {etiqueta} | {valor or '_(no disponible)_'} |" for etiqueta, valor in datos_lti.items())
tabla = f"| Dato | Valor |\n|---|---|\n{filas}"
display(Markdown(tabla))

## Todo lo disponible (debug)

Por si el `auth_state_hook` está exponiendo variables con otro prefijo, o quieres confirmar qué llegó exactamente sin depender de la lista fija de arriba. **Nunca imprimimos aquí nada que empiece con `OAUTH_` o contenga `SECRET`/`TOKEN`**, para no exponer accidentalmente material de firma del protocolo.

In [ ]:
PREFIJOS_A_MOSTRAR = ("ALUMNO_", "CURSO_")
PALABRAS_PROHIBIDAS = ("OAUTH", "SECRET", "TOKEN", "SIGNATURE")

variables_relevantes = {
    k: v for k, v in os.environ.items()
    if k.startswith(PREFIJOS_A_MOSTRAR)
    and not any(p in k.upper() for p in PALABRAS_PROHIBIDAS)
}

if not variables_relevantes:
    print("No hay variables ALUMNO_* / CURSO_* en el entorno de este kernel.")
else:
    for k, v in sorted(variables_relevantes.items()):
        print(f"{k} = {v}")

In [ ]:
from datetime import datetime

if "metricas_nbgrader" not in globals():
    metricas_nbgrader = {
        "intentos": 0,
        "errores": [],
        "inicio": datetime.now(),
        "fin": None,
        "puntaje": 0,
        "puntaje_maximo": 2,
        "estado": "incompleto",
    }

def _registrar_error(self, etype, value, tb, tb_offset=None):
    metricas_nbgrader["errores"].append({
        "tipo": etype.__name__,
        "mensaje": str(value),
        "hora": datetime.now().strftime("%H:%M:%S"),
    })
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)

get_ipython().set_custom_exc((Exception,), _registrar_error)

In [ ]:
# ============================================================
# EJERCICIO (solo para comprobar que todo funciona):
# Completa la función contar_vocales(palabra), que debe devolver
# cuántas vocales (a, e, i, o, u) tiene la palabra recibida.
# ============================================================
def contar_vocales(palabra):
    # ESCRIBE TU CÓDIGO AQUÍ
    raise NotImplementedError()

In [ ]:
metricas_nbgrader["intentos"] += 1

try:
    assert contar_vocales("programacion") == 5
    assert contar_vocales("xyz") == 0
    assert contar_vocales("") == 0
    metricas_nbgrader["puntaje"] = metricas_nbgrader["puntaje_maximo"]
    metricas_nbgrader["estado"] = "completado"
    metricas_nbgrader["fin"] = datetime.now()
    print("✅ ¡Ejercicio completado correctamente!")
except NotImplementedError:
    print("✏️ Todavía no has completado la función.")
except AssertionError:
    print("❌ Todavía no es correcto. Revisa tu función e inténtalo de nuevo.")
except Exception:
    print("❌ Tu función lanzó un error al ejecutarse. Revísala e inténtalo de nuevo.")

In [ ]:
from IPython.display import Markdown, display

duracion = None
if metricas_nbgrader.get("fin"):
    duracion = (metricas_nbgrader["fin"] - metricas_nbgrader["inicio"]).total_seconds()

filas_metricas = [
    ("Estado", metricas_nbgrader["estado"]),
    ("Puntaje obtenido", f'{metricas_nbgrader["puntaje"]} / {metricas_nbgrader["puntaje_maximo"]}'),
    ("Intentos", metricas_nbgrader["intentos"]),
    ("Errores propios capturados", len(metricas_nbgrader["errores"])),
    ("Duración hasta completar (segundos)", f"{duracion:.1f}" if duracion is not None else "(aún no termina)"),
]

filas_md = "\n".join(f"| {k} | {v} |" for k, v in filas_metricas)
display(Markdown(f"| Métrica | Valor |\n|---|---|\n{filas_md}"))

if metricas_nbgrader["errores"]:
    display(Markdown("### Errores que tuviste al ejecutar tu código"))
    for err in metricas_nbgrader["errores"]:
        display(Markdown(f"- `{err['hora']}` — **{err['tipo']}**: {err['mensaje']}"))